# Experimento: ¿mejora el Elo si aprende distinto?

Los experimentos anteriores agregaban **variables encima del Elo** (estilo, forma por rol, momentum). Este cambia
**cómo aprende el Elo** (`models/elo_learning.py`):

| Variante | Qué cambia | Parámetro |
|---|---|---|
| **Base** | El Elo actual: cada temporada arranca de cero (1500 + ajuste de la región) | — |
| **A. Arrastre** | Cada equipo arranca el año con una parte de lo que terminó por encima o por debajo de su región el año anterior | `carry` (0 = nada, 1 = todo) |
| **B. Región compartida** | En un game entre regiones, además de los dos equipos se mueve un rating que comparten todos los equipos de cada región | `pool` (0 = nada) |
| **A+B** | Las dos juntas | `carry` y `pool` |

**¿Por qué A?** Con K pequeño el Elo se mueve poco. Cada game "borra" más o menos K × ln(10)/400 × p(1−p) ≈ 1,4% del valor
inicial (con K = 10), así que después de 100 games todavía queda ~24% del punto de partida. Si ese punto de partida
es "el promedio de la región", para un equipo que viene de ganar Worlds se está tirando información.
Se arrastra solo **una parte** (regresión a la media) porque entre temporadas cambian rosters y meta.

**¿Por qué B?** Los ajustes regionales salieron inestables de un año a otro, y en Worlds casi todos los partidos
son entre regiones. Si los equipos de la LEC le ganan seguido a los de la LCK en el MSI, probablemente la LEC
entera esté mejor de lo que se creía, no solo esos equipos.

## Cómo se evalúa: validación *walk-forward*

Para cada año de prueba T (2022 a 2026):

1. Se ajustan los parámetros de cada variante **solo con T−2 y T−1** (K, ajustes regionales, `carry`, `pool`).
2. Con esos parámetros se predice T, game por game, **antes** de conocer cada resultado.

Es lo que pasaría en la realidad: ajustar con el pasado y predecir lo que viene. Da **5 pruebas fuera de la muestra**
en vez de una sola, y sirve para ver si la mejora se repite o aparece un año por casualidad.
2019 solo sirve de "calentamiento": es el año del que arrastra 2020 y nunca se evalúa.

Todas las variantes se ajustan igual (misma grilla, mismo objetivo: Brier de todos los games de los dos años).
La Fase 1 usa solo el Elo, sin la forma por rol: los CSV de Oracle's Elixir anteriores a 2024 no se pudieron bajar
(Google Drive limitó las descargas) y A y B cambian solo la parte del Elo. La Fase 2 (sección 5) agrega la forma por
rol encima con los años que sí hay (2024–2026).

## Regla de decisión (fijada antes de mirar los resultados)

Una variante **mejora** si cumple las tres:

1. **Todos los games de 2022–2026 juntos:** el IC del 95% de su diferencia de Brier contra Base está entero por debajo de 0.
2. **Consistencia:** mejora (diferencia < 0) en al menos 4 de los 5 años.
3. **Entre regiones (2022–2026 juntos):** no empeora de forma detectable (el IC no está entero por encima de 0).

A+B se prefiere sobre una sola solo si además mejora a esa sola con IC entero por debajo de 0: si no, se queda la más simple.
Si alguna pasa, el siguiente paso es confirmarla con la forma por rol encima (modelo completo) antes de pasarla a producción.

⚠️ **Por qué tanta regla:** este es el cuarto experimento sobre el modelo. Cuantas más variantes probás, más fácil es
que alguna "mejore" por suerte (*comparaciones múltiples*). Fijar la regla antes y pedir que se repita en varios años
es la defensa.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from models.backtest import K_GRID, OFFSET_GRID, TUNED_REGIONS, cross_region_records, worlds_series
from models.elo import PARAMS, team_regions
from models.elo_learning import LearningParams, run_chain
from models.metrics import accuracy, bootstrap_brier_diff, brier_score, log_loss
from pipeline.history import load_aliases, load_season_games

RAW = Path.cwd().parent / "data" / "raw"
ALIASES = load_aliases(Path.cwd().parent / "data" / "teams.csv")
YEARS = list(range(2019, 2027))
TEST_YEARS = [2022, 2023, 2024, 2025, 2026]

seasons = {}
for y in YEARS:
    games = load_season_games(RAW, y, ALIASES)
    seasons[y] = (games, team_regions(games))

## 1. Los datos

Games de ligas principales e internacionales de Leaguepedia, por año. "Entre regiones" son los games de equipos de
regiones distintas: los que ponen a prueba los ajustes regionales y la variante B. Son pocos, así que sus
resultados van a tener intervalos anchos.

In [2]:
def count_row(y):
    games, regions = seasons[y]
    cross = sum(1 for g in games if regions.get(g["blue"]) and regions.get(g["red"])
                and regions[g["blue"]] != regions[g["red"]])
    return {"año": y, "games": len(games), "entre regiones": cross,
            "internacionales": sum(1 for g in games if g["region"] is None),
            "equipos": len(regions)}

pd.DataFrame([count_row(y) for y in YEARS]).set_index("año")

,games,entre regiones,internacionales,equipos
año,,,,
2019,3550,156,219,133
2020,3471,103,136,126
2021,3689,189,224,123
2022,3701,190,232,123
2023,3411,163,198,105
2024,3083,255,229,88
2025,2608,217,284,64
2026,2552,143,179,62


¿Cuántos equipos de cada año ya estaban el año anterior (mismo nombre)? Si fueran pocos, A no tendría de dónde arrastrar.
Un cambio de nombre cuenta como equipo nuevo y arranca en el valor de su región (lo prudente).

In [3]:
rows = []
for y in YEARS[1:]:
    now, before = set(seasons[y][1]), set(seasons[y - 1][1])
    rows.append({"año": y, "equipos": len(now), "ya estaban": len(now & before),
                 "%": round(len(now & before) / len(now) * 100)})
pd.DataFrame(rows).set_index("año")

,equipos,ya estaban,%
año,,,
2020,126,93,74
2021,123,94,76
2022,123,102,83
2023,105,82,78
2024,88,75,85
2025,64,52,81
2026,62,50,81


## 2. Ajuste de parámetros

Búsqueda por coordenadas, igual que en `models/backtest.py`: para cada parámetro se prueban todos los valores de su
grilla con los demás quietos, y se repite hasta que una vuelta entera no mejora nada. La diferencia con el backtest
original es el objetivo: acá es el Brier de **todos** los games (A cambia todos los games, no solo los cruces).
Los ajustes regionales igual quedan determinados solo por los cruces, porque dentro de una región se cancelan.

In [4]:
# Grillas más amplias que las de models/backtest.py: en una primera corrida K y algunos ajustes
# regionales quedaron en el borde (40 y 600), y un óptimo en el borde puede estar más allá
K_GRID = K_GRID + (50, 60, 80, 100)
OFFSET_GRID = tuple(range(-100, 901, 50))
CARRY_GRID = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
POOL_GRID = (0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0)
VARIANTS = {"Base": (), "A": ("carry",), "B": ("pool",), "A+B": ("carry", "pool")}


def chain_records(params, first, last):
    # Corre desde 2019 (calentamiento) hasta `last` y devuelve los registros por año desde `first`
    years = list(range(YEARS[0], last + 1))
    runs = run_chain([seasons[y] for y in years], params)
    return {y: run.records for y, run in zip(years, runs) if y >= first}


def objective(params, years):
    recs = chain_records(params, years[0], years[-1])
    probs = [r["p_blue"] for y in years for r in recs[y]]
    outcomes = [r["blue_won"] for y in years for r in recs[y]]
    return brier_score(probs, outcomes)


def with_value(p, name, value):
    if name in ("k", "carry", "pool"):
        return LearningParams(**{**p.__dict__, name: float(value)})
    return LearningParams(**{**p.__dict__, "region_offsets": {**p.region_offsets, name: float(value)}})


def tune(extra, years, max_sweeps=5):
    grids = {"k": K_GRID, **{r: OFFSET_GRID for r in TUNED_REGIONS}, "carry": CARRY_GRID, "pool": POOL_GRID}
    names = ("k",) + TUNED_REGIONS + extra
    best = LearningParams(k=20.0)
    best_score = objective(best, years)
    for _ in range(max_sweeps):
        improved = False
        for name in names:
            for value in grids[name]:
                candidate = with_value(best, name, value)
                score = objective(candidate, years)
                if score < best_score - 1e-9:
                    best, best_score, improved = candidate, score, True
        if not improved:
            break
    return best, best_score


start = time.time()
tuned = {}
for T in TEST_YEARS:
    for name, extra in VARIANTS.items():
        tuned[(T, name)] = tune(extra, [T - 2, T - 1])
print(f"ajuste: {time.time() - start:.0f} s")

ajuste: 492 s


### Parámetros elegidos en cada ventana

Si un parámetro cae en el borde de su grilla, el óptimo real podría estar más allá. Y si salta mucho de una ventana a
otra, el dato está **poco determinado** (muestra chica): conviene leerlo como aproximado.

In [5]:
rows = []
for (T, name), (p, score) in tuned.items():
    rows.append({"prueba": T, "ajustado con": f"{T-2}–{T-1}", "variante": name, "K": p.k,
                 **{r: p.region_offsets.get(r, 0.0) for r in TUNED_REGIONS},
                 "carry": p.carry, "pool": p.pool, "Brier ajuste": round(score, 4)})
params_table = pd.DataFrame(rows)
params_table

,prueba,ajustado con,variante,K,KR,CN,EU,NA,carry,pool,Brier ajuste
0,2022,2020–2021,Base,40.0,450.0,400.0,300.0,250.0,0.0,0.0,0.2243
1,2022,2020–2021,A,30.0,450.0,400.0,250.0,250.0,0.5,0.0,0.2229
2,2022,2020–2021,B,40.0,450.0,400.0,300.0,250.0,0.0,0.0,0.2243
3,2022,2020–2021,A+B,30.0,450.0,400.0,250.0,250.0,0.5,0.0,0.2229
4,2023,2021–2022,Base,40.0,550.0,450.0,350.0,250.0,0.0,0.0,0.2184
5,2023,2021–2022,A,30.0,550.0,500.0,350.0,250.0,0.6,0.0,0.2159
6,2023,2021–2022,B,40.0,650.0,550.0,400.0,300.0,0.0,0.0,0.2183
7,2023,2021–2022,A+B,30.0,600.0,500.0,350.0,250.0,0.6,0.0,0.2159
8,2024,2022–2023,Base,40.0,650.0,600.0,450.0,300.0,0.0,0.0,0.2173
9,2024,2022–2023,A,30.0,650.0,600.0,450.0,350.0,0.6,0.0,0.2140


## 3. Resultados fuera de la muestra

Para cada año de prueba se predice con los parámetros ajustados en los dos años anteriores.
La columna **Δ vs Base** es la diferencia de Brier (negativo = mejor que Base) con su IC del 95% por bootstrap.
El bootstrap sortea **series enteras**, porque los games de una misma serie no son independientes.

In [6]:
preds = {}
for (T, name), (p, _) in tuned.items():
    preds[(T, name)] = chain_records(p, T, T)[T]
# Referencia: el Elo de producción tal cual (K = 10 y ajustes de models/elo.py), sin reajustar.
# ⚠️ Se eligió mirando 2024 y 2025: en esos años juega con ventaja. No entra en la regla.
PROD = LearningParams(k=PARAMS.k, region_offsets=PARAMS.region_offsets)
for T in TEST_YEARS:
    preds[(T, "Producción")] = chain_records(PROD, T, T)[T]


def subset(records, T, which):
    regions = seasons[T][1]
    if which == "todos":
        return records
    if which == "entre regiones":
        return cross_region_records(records, regions)
    return [r for r in records if r["region"] is None]  # internacionales


def compare(which, years):
    rows = []
    base = [r for T in years for r in subset(preds[(T, "Base")], T, which)]
    outcomes = [r["blue_won"] for r in base]
    clusters = [r["match_id"] for r in base]
    p_base = [r["p_blue"] for r in base]
    for name in (*VARIANTS, "Producción"):
        recs = [r for T in years for r in subset(preds[(T, name)], T, which)]
        p = [r["p_blue"] for r in recs]
        ci = bootstrap_brier_diff(p, p_base, outcomes, clusters, reps=2000)
        rows.append({"games": len(p), "variante": name, "brier": round(brier_score(p, outcomes), 4),
                     "log_loss": round(log_loss(p, outcomes), 4), "precisión": round(accuracy(p, outcomes), 3),
                     "Δ vs Base": ci["diff"], "IC 95%": f"[{ci['low']:+.4f}, {ci['high']:+.4f}]",
                     "IC bajo": ci["low"], "IC alto": ci["high"]})
    return rows


results = []
for which in ("todos", "entre regiones", "internacionales"):
    for T in TEST_YEARS:
        results += [{"partidos": which, "prueba": str(T), **row} for row in compare(which, [T])]
    results += [{"partidos": which, "prueba": "2022–2026", **row} for row in compare(which, TEST_YEARS)]
results = pd.DataFrame(results)
show = ["partidos", "prueba", "games", "variante", "brier", "log_loss", "precisión", "Δ vs Base", "IC 95%"]
results[results.partidos == "todos"][show]

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs Base,IC 95%
0,todos,2022,3701,Base,0.2151,0.6181,0.656,0.000000,"[+0.0000, +0.0000]"
1,todos,2022,3701,A,0.2128,0.6130,0.659,-0.002264,"[-0.0037, -0.0008]"
2,todos,2022,3701,B,0.2151,0.6181,0.656,0.000000,"[+0.0000, +0.0000]"
3,todos,2022,3701,A+B,0.2128,0.6130,0.659,-0.002264,"[-0.0037, -0.0008]"
4,todos,2022,3701,Producción,0.2257,0.6428,0.658,0.010591,"[+0.0078, +0.0132]"
5,todos,2023,3411,Base,0.2203,0.6310,0.644,0.000000,"[+0.0000, +0.0000]"
6,todos,2023,3411,A,0.2161,0.6215,0.657,-0.004217,"[-0.0059, -0.0025]"
7,todos,2023,3411,B,0.2202,0.6306,0.644,-0.000139,"[-0.0003, +0.0000]"
8,todos,2023,3411,A+B,0.2161,0.6215,0.656,-0.004193,"[-0.0059, -0.0025]"
9,todos,2023,3411,Producción,0.2285,0.6489,0.645,0.008214,"[+0.0053, +0.0110]"


In [7]:
results[results.partidos == "entre regiones"][show]

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs Base,IC 95%
30,entre regiones,2022,190,Base,0.1755,0.5232,0.732,0.000000,"[+0.0000, +0.0000]"
31,entre regiones,2022,190,A,0.1731,0.5169,0.747,-0.002361,"[-0.0069, +0.0019]"
32,entre regiones,2022,190,B,0.1755,0.5232,0.732,0.000000,"[+0.0000, +0.0000]"
33,entre regiones,2022,190,A+B,0.1731,0.5169,0.747,-0.002361,"[-0.0069, +0.0019]"
34,entre regiones,2022,190,Producción,0.1788,0.5371,0.726,0.003326,"[-0.0096, +0.0159]"
35,entre regiones,2023,163,Base,0.2117,0.6090,0.607,0.000000,"[+0.0000, +0.0000]"
36,entre regiones,2023,163,A,0.2078,0.5984,0.620,-0.003885,"[-0.0105, +0.0026]"
37,entre regiones,2023,163,B,0.2093,0.6025,0.613,-0.002358,"[-0.0055, +0.0007]"
38,entre regiones,2023,163,A+B,0.2086,0.5990,0.620,-0.003148,"[-0.0097, +0.0036]"
39,entre regiones,2023,163,Producción,0.2200,0.6271,0.607,0.008273,"[-0.0074, +0.0242]"


In [8]:
results[results.partidos == "internacionales"][show]

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs Base,IC 95%
60,internacionales,2022,232,Base,0.1974,0.5718,0.664,0.000000,"[+0.0000, +0.0000]"
61,internacionales,2022,232,A,0.1950,0.5658,0.694,-0.002380,"[-0.0062, +0.0016]"
62,internacionales,2022,232,B,0.1974,0.5718,0.664,0.000000,"[+0.0000, +0.0000]"
63,internacionales,2022,232,A+B,0.1950,0.5658,0.694,-0.002380,"[-0.0062, +0.0016]"
64,internacionales,2022,232,Producción,0.1965,0.5755,0.677,-0.000859,"[-0.0116, +0.0104]"
65,internacionales,2023,198,Base,0.2046,0.5955,0.662,0.000000,"[+0.0000, +0.0000]"
66,internacionales,2023,198,A,0.2030,0.5899,0.662,-0.001573,"[-0.0056, +0.0030]"
67,internacionales,2023,198,B,0.2023,0.5895,0.667,-0.002264,"[-0.0047, +0.0004]"
68,internacionales,2023,198,A+B,0.2037,0.5906,0.662,-0.000888,"[-0.0050, +0.0035]"
69,internacionales,2023,198,Producción,0.2143,0.6165,0.662,0.009699,"[-0.0031, +0.0223]"


### Series de Worlds

Lo que más importa para el sitio. La probabilidad de la serie sale del primer game (la que había antes de empezar).
Son ~40–90 series por año: sirve para ver la tendencia, no para decidir.
Worlds 2026 todavía no tiene series terminadas en los datos.

In [9]:
def schedule_rows(T):
    import json
    path = RAW / f"worlds_{T}_matches.json"
    return json.loads(path.read_text(encoding="utf-8"))["rows"] if path.exists() else []


rows = []
for T in TEST_YEARS:
    sched = schedule_rows(T)
    if not any(r.get("Winner") in ("1", "2") for r in sched):
        continue
    for name in (*VARIANTS, "Producción"):
        probs, outcomes = worlds_series(preds[(T, name)], sched)
        rows.append({"Worlds": T, "variante": name, "series": len(probs),
                     "brier": round(brier_score(probs, outcomes), 4) if probs else None,
                     "aciertos": round(accuracy(probs, outcomes), 3) if probs else None})
series_table = pd.DataFrame(rows)
series_table.pivot(index="Worlds", columns="variante", values="brier")

variante,A,A+B,B,Base,Producción
Worlds,,,,,
2022,0.1800,0.1800,0.1833,0.1833,0.1802
2023,0.1982,0.1981,0.1986,0.2001,0.2010
2024,0.1756,0.1997,0.1994,0.1700,0.1947
2025,0.1886,0.1763,0.1796,0.1891,0.1710


## 4. La regla de decisión

In [10]:
def verdict(name):
    pooled = results[(results.partidos == "todos") & (results.prueba == "2022–2026") & (results.variante == name)].iloc[0]
    yearly = results[(results.partidos == "todos") & (results.prueba != "2022–2026") & (results.variante == name)]
    cross = results[(results.partidos == "entre regiones") & (results.prueba == "2022–2026")
                    & (results.variante == name)].iloc[0]
    c1 = pooled["IC alto"] < 0
    years_better = int((yearly["Δ vs Base"] < 0).sum())
    c2 = years_better >= 4
    c3 = not cross["IC bajo"] > 0
    return {"variante": name, "1. IC todos < 0": c1, f"2. años que mejora": f"{years_better}/5",
            "2. ≥ 4/5": c2, "3. no empeora cruces": c3, "PASA": c1 and c2 and c3}

pd.DataFrame([verdict(n) for n in ("A", "B", "A+B")])

,variante,1. IC todos < 0,2. años que mejora,2. ≥ 4/5,3. no empeora cruces,PASA
0,A,True,5/5,True,True,True
1,B,False,1/5,False,False,False
2,A+B,True,5/5,True,True,True


A+B contra cada variante sola (todos los games 2022–2026). Solo importa si A+B pasa la regla.

In [11]:
def pooled_vs(name_a, name_b, which="todos"):
    a = [r for T in TEST_YEARS for r in subset(preds[(T, name_a)], T, which)]
    b = [r for T in TEST_YEARS for r in subset(preds[(T, name_b)], T, which)]
    return bootstrap_brier_diff([r["p_blue"] for r in a], [r["p_blue"] for r in b],
                                [r["blue_won"] for r in a], [r["match_id"] for r in a], reps=2000)

pd.DataFrame([{"comparación": f"A+B vs {n}", **pooled_vs("A+B", n)} for n in ("A", "B")])

,comparación,diff,low,high
0,A+B vs A,0.000239,-0.000003,0.000483
1,A+B vs B,-0.003477,-0.004401,-0.002545


## 5. Fase 2: con la forma por rol encima (el modelo completo)

El sitio no usa el Elo solo: usa **elo-2** = Elo + forma de los jugadores por rol. Puede pasar que A y la forma por rol
capten lo mismo (las dos dicen "este equipo es mejor de lo que el Elo cree al principio del año") y que la mejora
desaparezca al combinarlas. Por eso se prueba el modelo completo.

Datos: Oracle's Elixir 2024–2026 (los únicos años que se pudieron bajar). Pruebas: **2025 y 2026**.
- Elo de cada variante con los parámetros ya ajustados en la Fase 1 para ese año (sin mirar el año de prueba).
- El arrastre hacia 2024 sale del Elo de Leaguepedia de 2023 (algunos equipos tienen otro nombre en Oracle's Elixir: tabla abajo).
- La forma por rol se reajusta encima de cada Elo con los años anteriores a la prueba (2024 para 2025; 2024–2025 para 2026),
  igual que en producción.

**Regla de la Fase 2 (fijada antes de correrla):** A se confirma si, con la forma por rol encima y 2025–2026 juntos,
1. mejora a Base + roles con IC entero por debajo de 0 (todos los games), y
2. no empeora de forma detectable los games internacionales (IC no entero por encima de 0).

También se muestra **Producción (elo-2)** tal cual, con la advertencia de siempre: su Elo se eligió mirando 2024 y 2025,
así que en 2025 juega con ventaja.

In [12]:
from math import exp

from models.elo import series_probability
from models.elo_learning import run_season
from models.features import ROLE_FEATURES, build_dataset, fit_norms
from models.oracles import load_season
from models.role_form import fit as fit_roles

OE_DIR = RAW / "oracles_elixir"
OE_YEARS = (2024, 2025, 2026)
oe = {y: load_season(OE_DIR / f"{y}.csv") for y in OE_YEARS}
oe_regions = {y: team_regions(oe[y]) for y in OE_YEARS}

# Nombre en Oracle's Elixir → nombre en Leaguepedia (solo los que difieren)
OE_TO_LP = {"DN SOOPers": "Kwangdong Freecs", "HANJIN BRION": "OKSavingsBank BRION", "Kiwoom DRX": "DRX",
            "Ninjas in Pyjamas": "Ninjas in Pyjamas.CN", "GiantX": "GIANTX", "Team Flash": "Team Flash.Vietnam",
            "MGN Vikings Esports": "Vikings Esports (2023 Vietnamese Team)",
            "West Point Esports": "West Point Esports Philippines", "inf": "INFINITY"}


def oe_records(params, last):
    # Arrastre hacia 2024: Elo de Leaguepedia 2019–2023 con los mismos parámetros
    dev = run_chain([seasons[y] for y in range(YEARS[0], 2024)], params)[-1].final_deviation
    prior = {t: dev.get(OE_TO_LP.get(t, t), 0.0) for t in oe_regions[2024]}
    out = {}
    for y in OE_YEARS:
        if y > last:
            break
        run = run_season(oe[y], params, oe_regions[y], prior)
        out[y], prior = run.records, run.final_deviation
    return out


def full_model(params, T):
    # Elo de la variante + forma por rol ajustada con los años anteriores a T. Devuelve las filas de T
    train_years = [y for y in OE_YEARS if y < T]
    recs = oe_records(params, T)
    norms = fit_norms([g for y in train_years for g in oe[y]])
    weights, scales = fit_roles([recs[y] for y in train_years], norms)
    rows = []
    for row, rec in zip(build_dataset(recs[T], norms), recs[T]):
        z = row["elo_logit"] + sum(weights[c[2:]] * row[c] / scales[c[2:]] for c in ROLE_FEATURES)
        rows.append({**{k: rec[k] for k in ("blue", "red", "region", "page", "match_id", "blue_won")},
                     "p_elo": rec["p_blue"], "p_blue": 1 / (1 + exp(-z))})
    return rows


FULL_TESTS = (2025, 2026)
FULL_VARIANTS = ("Producción (elo-2)", "Base + roles", "A + roles")
full = {}
for T in FULL_TESTS:
    full[(T, "Producción (elo-2)")] = full_model(PROD, T)
    full[(T, "Base + roles")] = full_model(tuned[(T, "Base")][0], T)
    full[(T, "A + roles")] = full_model(tuned[(T, "A")][0], T)

In [13]:
def full_subset(rows, T, which):
    if which == "todos":
        return rows
    if which == "internacionales":
        return [r for r in rows if r["region"] is None]
    regions = oe_regions[T]
    return [r for r in rows if regions.get(r["blue"]) and regions.get(r["red"])
            and regions[r["blue"]] != regions[r["red"]]]


def full_compare(which, years, reference="Base + roles"):
    ref = [r for T in years for r in full_subset(full[(T, reference)], T, which)]
    outcomes, clusters = [r["blue_won"] for r in ref], [r["match_id"] for r in ref]
    out = []
    for name in FULL_VARIANTS:
        rows = [r for T in years for r in full_subset(full[(T, name)], T, which)]
        p = [r["p_blue"] for r in rows]
        ci = bootstrap_brier_diff(p, [r["p_blue"] for r in ref], outcomes, clusters, reps=2000)
        out.append({"games": len(p), "variante": name, "brier": round(brier_score(p, outcomes), 4),
                    "log_loss": round(log_loss(p, outcomes), 4), "precisión": round(accuracy(p, outcomes), 3),
                    "Δ vs ref": ci["diff"], "IC 95%": f"[{ci['low']:+.4f}, {ci['high']:+.4f}]",
                    "IC bajo": ci["low"], "IC alto": ci["high"]})
    return out


PERIODS = (("2025", [2025]), ("2026", [2026]), ("2025–2026", [2025, 2026]))
full_results = []
for which in ("todos", "entre regiones", "internacionales"):
    for label, years in PERIODS:
        full_results += [{"partidos": which, "prueba": label, **r} for r in full_compare(which, years)]
full_results = pd.DataFrame(full_results)
full_results.drop(columns=["IC bajo", "IC alto"])

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs ref,IC 95%
0,todos,2025,3496,Producción (elo-2),0.2194,0.6286,0.657,0.001617,"[-0.0006, +0.0040]"
1,todos,2025,3496,Base + roles,0.2177,0.6255,0.652,0.000000,"[+0.0000, +0.0000]"
2,todos,2025,3496,A + roles,0.2157,0.6210,0.656,-0.002002,"[-0.0044, +0.0002]"
3,todos,2026,3367,Producción (elo-2),0.2237,0.6390,0.635,0.000604,"[-0.0018, +0.0031]"
4,todos,2026,3367,Base + roles,0.2231,0.6382,0.638,0.000000,"[+0.0000, +0.0000]"
5,todos,2026,3367,A + roles,0.2205,0.6324,0.642,-0.002640,"[-0.0048, -0.0005]"
6,todos,2025–2026,6863,Producción (elo-2),0.2215,0.6337,0.646,0.001120,"[-0.0005, +0.0028]"
7,todos,2025–2026,6863,Base + roles,0.2204,0.6317,0.645,0.000000,"[+0.0000, +0.0000]"
8,todos,2025–2026,6863,A + roles,0.2180,0.6266,0.649,-0.002315,"[-0.0039, -0.0007]"
9,entre regiones,2025,234,Producción (elo-2),0.2138,0.6193,0.684,-0.019894,"[-0.0354, -0.0062]"


### Contra producción

A + roles tomando como referencia el modelo que está hoy en el sitio.

In [14]:
vs_prod = []
for which in ("todos", "entre regiones", "internacionales"):
    for label, years in PERIODS:
        rows = full_compare(which, years, reference="Producción (elo-2)")
        vs_prod += [{"partidos": which, "prueba": label, **r} for r in rows if r["variante"] == "A + roles"]
pd.DataFrame(vs_prod).drop(columns=["IC bajo", "IC alto"])

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs ref,IC 95%
0,todos,2025,3496,A + roles,0.2157,0.6210,0.656,-0.003619,"[-0.0067, -0.0007]"
1,todos,2026,3367,A + roles,0.2205,0.6324,0.642,-0.003244,"[-0.0064, -0.0001]"
2,todos,2025–2026,6863,A + roles,0.2180,0.6266,0.649,-0.003435,"[-0.0056, -0.0013]"
3,entre regiones,2025,234,A + roles,0.2249,0.6483,0.662,0.011187,"[-0.0014, +0.0253]"
4,entre regiones,2026,152,A + roles,0.1918,0.5591,0.671,0.009248,"[-0.0048, +0.0244]"
5,entre regiones,2025–2026,386,A + roles,0.2119,0.6132,0.666,0.010424,"[+0.0004, +0.0202]"
6,internacionales,2025,284,A + roles,0.2304,0.6589,0.658,0.010112,"[-0.0006, +0.0213]"
7,internacionales,2026,360,A + roles,0.2038,0.5888,0.667,0.004036,"[-0.0047, +0.0130]"
8,internacionales,2025–2026,644,A + roles,0.2156,0.6197,0.663,0.006716,"[-0.0003, +0.0137]"


### Series: Worlds 2025 y MSI 2026

El formato de cada serie se deduce de los games ganados (2 → Bo3, 3 → Bo5).

In [15]:
def series_brier(rows):
    first, wins = {}, {}
    for r in rows:
        first.setdefault(r["match_id"], r)
        winner = r["blue"] if r["blue_won"] == 1 else r["red"]
        wins.setdefault(r["match_id"], {}).setdefault(winner, 0)
        wins[r["match_id"]][winner] += 1
    probs, outcomes = [], []
    for mid, r in first.items():
        winner, n = max(wins[mid].items(), key=lambda kv: kv[1])
        probs.append(series_probability(r["p_blue"], 2 * n - 1))
        outcomes.append(1.0 if winner == r["blue"] else 0.0)
    return len(probs), round(brier_score(probs, outcomes), 4), round(accuracy(probs, outcomes), 3)


rows = []
for T, page, label in ((2025, "WLDs", "Worlds 2025"), (2026, "MSI", "MSI 2026")):
    for name in FULL_VARIANTS:
        n, brier, acc = series_brier([r for r in full[(T, name)] if r["page"] == page])
        rows.append({"torneo": label, "variante": name, "series": n, "brier": brier, "aciertos": acc})
pd.DataFrame(rows).pivot(index="variante", columns="torneo", values=["series", "brier", "aciertos"])

series                brier             aciertos  \
torneo             MSI 2026 Worlds 2025 MSI 2026 Worlds 2025 MSI 2026   
variante                                                                
A + roles              20.0        44.0   0.2044      0.1859     0.65   
Base + roles           20.0        44.0   0.2045      0.1875     0.65   
Producción (elo-2)     20.0        44.0   0.1569      0.1842     0.80   

                                
torneo             Worlds 2025  
variante                        
A + roles                0.795  
Base + roles             0.727  
Producción (elo-2)       0.727

In [16]:
def full_verdict():
    pick = lambda which: full_results[(full_results.partidos == which) & (full_results.prueba == "2025–2026")
                                      & (full_results.variante == "A + roles")].iloc[0]
    c1 = pick("todos")["IC alto"] < 0
    c2 = not pick("internacionales")["IC bajo"] > 0
    return {"1. IC todos < 0": c1, "2. no empeora internacionales": c2, "CONFIRMA": c1 and c2}

pd.DataFrame([full_verdict()])

,1. IC todos < 0,2. no empeora internacionales,CONFIRMA
0,True,True,True


## 6. Variante post-hoc: A ajustada como producción

La Fase 2 muestra una tensión: A + roles le gana a Base + roles, pero **contra producción empeora los cruces y los
internacionales**. La diferencia está en K: acá se ajustó con el Brier de todos los games y eso elige K = 30–40 (un Elo
que persigue la forma, útil dentro de las ligas). Producción se ajustó con los cruces entre regiones y eligió K = 10
(un Elo estable). Para el sitio importan los internacionales: el objetivo de ajuste no coincidía con el uso.

Variante nueva: **Base y A ajustadas con el Brier de los cruces entre regiones** (el mismo objetivo que producción).

⚠️ **Post-hoc:** se diseñó después de ver los resultados. Por eso tiene su propia regla, fijada antes de correrla.
Se adopta si cumple las tres:
1. **Fase 1, cruces 2022–2026 juntos:** mejora a Base (cruces) con IC entero por debajo de 0.
2. **Consistencia:** mejora los cruces en al menos 4 de los 5 años.
3. **Fase 2, internacionales 2025–2026:** A (cruces) + roles mejora a Producción (diferencia < 0) sin que el IC quede entero por encima de 0.

In [17]:
def objective_cross(params, years):
    recs = chain_records(params, years[0], years[-1])
    sel = [r for y in years for r in cross_region_records(recs[y], seasons[y][1])]
    return brier_score([r["p_blue"] for r in sel], [r["blue_won"] for r in sel])


def tune_cross(extra, years, max_sweeps=5):
    # Igual que tune(), pero el objetivo son solo los cruces entre regiones
    grids = {"k": K_GRID, **{r: OFFSET_GRID for r in TUNED_REGIONS}, "carry": CARRY_GRID}
    names = ("k",) + TUNED_REGIONS + extra
    best = LearningParams(k=20.0)
    best_score = objective_cross(best, years)
    for _ in range(max_sweeps):
        improved = False
        for name in names:
            for value in grids[name]:
                candidate = with_value(best, name, value)
                score = objective_cross(candidate, years)
                if score < best_score - 1e-9:
                    best, best_score, improved = candidate, score, True
        if not improved:
            break
    return best, best_score


CROSS_VARIANTS = {"Base (cruces)": (), "A (cruces)": ("carry",)}
for T in TEST_YEARS:
    for name, extra in CROSS_VARIANTS.items():
        tuned[(T, name)] = tune_cross(extra, [T - 2, T - 1])
        preds[(T, name)] = chain_records(tuned[(T, name)][0], T, T)[T]

pd.DataFrame([{"prueba": T, "variante": name, "K": tuned[(T, name)][0].k,
               **{r: tuned[(T, name)][0].region_offsets.get(r, 0.0) for r in TUNED_REGIONS},
               "carry": tuned[(T, name)][0].carry} for T in TEST_YEARS for name in CROSS_VARIANTS])

,prueba,variante,K,KR,CN,EU,NA,carry
0,2022,Base (cruces),10.0,300.0,250.0,150.0,150.0,0.0
1,2022,A (cruces),10.0,350.0,300.0,200.0,150.0,0.4
2,2023,Base (cruces),6.0,400.0,350.0,200.0,150.0,0.0
3,2023,A (cruces),6.0,400.0,350.0,200.0,150.0,0.1
4,2024,Base (cruces),4.0,450.0,400.0,200.0,150.0,0.0
5,2024,A (cruces),4.0,450.0,400.0,200.0,150.0,0.0
6,2025,Base (cruces),4.0,350.0,350.0,200.0,150.0,0.0
7,2025,A (cruces),4.0,300.0,300.0,150.0,150.0,0.7
8,2026,Base (cruces),10.0,300.0,200.0,100.0,100.0,0.0
9,2026,A (cruces),6.0,300.0,200.0,100.0,150.0,0.7


In [18]:
def pair_compare(name, ref, which, years):
    a = [r for T in years for r in subset(preds[(T, name)], T, which)]
    b = [r for T in years for r in subset(preds[(T, ref)], T, which)]
    ci = bootstrap_brier_diff([r["p_blue"] for r in a], [r["p_blue"] for r in b],
                              [r["blue_won"] for r in a], [r["match_id"] for r in a], reps=2000)
    return {"games": len(a), "brier": round(brier_score([r["p_blue"] for r in a], [r["blue_won"] for r in a]), 4),
            "Δ vs Base (cruces)": ci["diff"], "IC 95%": f"[{ci['low']:+.4f}, {ci['high']:+.4f}]",
            "IC bajo": ci["low"], "IC alto": ci["high"]}


cross_rows = []
for which in ("entre regiones", "internacionales", "todos"):
    for label, years in [(str(T), [T]) for T in TEST_YEARS] + [("2022–2026", TEST_YEARS)]:
        for name in ("A (cruces)", "Producción"):
            cross_rows.append({"partidos": which, "prueba": label, "variante": name,
                               **pair_compare(name, "Base (cruces)", which, years)})
cross_results = pd.DataFrame(cross_rows)
cross_results.drop(columns=["IC bajo", "IC alto"])

,partidos,prueba,variante,games,brier,Δ vs Base (cruces),IC 95%
0,entre regiones,2022,A (cruces),190,0.1690,-0.006262,"[-0.0108, -0.0017]"
1,entre regiones,2022,Producción,190,0.1788,0.003545,"[-0.0009, +0.0082]"
2,entre regiones,2023,A (cruces),163,0.2059,0.000584,"[-0.0003, +0.0015]"
3,entre regiones,2023,Producción,163,0.2200,0.014697,"[+0.0062, +0.0243]"
4,entre regiones,2024,A (cruces),255,0.2226,0.000000,"[+0.0000, +0.0000]"
5,entre regiones,2024,Producción,255,0.2234,0.000780,"[-0.0084, +0.0103]"
6,entre regiones,2025,A (cruces),217,0.2193,-0.004584,"[-0.0105, +0.0011]"
7,entre regiones,2025,Producción,217,0.2065,-0.017429,"[-0.0289, -0.0065]"
8,entre regiones,2026,A (cruces),143,0.1885,-0.004267,"[-0.0115, +0.0026]"
9,entre regiones,2026,Producción,143,0.1928,0.000000,"[+0.0000, +0.0000]"


In [19]:
for T in FULL_TESTS:
    full[(T, "Base (cruces) + roles")] = full_model(tuned[(T, "Base (cruces)")][0], T)
    full[(T, "A (cruces) + roles")] = full_model(tuned[(T, "A (cruces)")][0], T)

FULL_VARIANTS = ("Producción (elo-2)", "Base (cruces) + roles", "A (cruces) + roles", "A + roles")
full_cross = []
for which in ("internacionales", "entre regiones", "todos"):
    for label, years in PERIODS:
        full_cross += [{"partidos": which, "prueba": label, **r}
                       for r in full_compare(which, years, reference="Producción (elo-2)")]
full_cross = pd.DataFrame(full_cross)
full_cross.drop(columns=["IC bajo", "IC alto"])

,partidos,prueba,games,variante,brier,log_loss,precisión,Δ vs ref,IC 95%
0,internacionales,2025,284,Producción (elo-2),0.2203,0.6330,0.665,0.000000,"[+0.0000, +0.0000]"
1,internacionales,2025,284,Base (cruces) + roles,0.2308,0.6586,0.620,0.010524,"[+0.0024, +0.0188]"
2,internacionales,2025,284,A (cruces) + roles,0.2293,0.6540,0.648,0.008945,"[+0.0022, +0.0162]"
3,internacionales,2025,284,A + roles,0.2304,0.6589,0.658,0.010112,"[-0.0006, +0.0213]"
4,internacionales,2026,360,Producción (elo-2),0.1998,0.5848,0.683,0.000000,"[+0.0000, +0.0000]"
5,internacionales,2026,360,Base (cruces) + roles,0.1998,0.5848,0.683,0.000000,"[+0.0000, +0.0000]"
6,internacionales,2026,360,A (cruces) + roles,0.1977,0.5800,0.683,-0.002109,"[-0.0065, +0.0024]"
7,internacionales,2026,360,A + roles,0.2038,0.5888,0.667,0.004036,"[-0.0047, +0.0130]"
8,internacionales,2025–2026,644,Producción (elo-2),0.2088,0.6061,0.675,0.000000,"[+0.0000, +0.0000]"
9,internacionales,2025–2026,644,Base (cruces) + roles,0.2135,0.6174,0.655,0.004641,"[+0.0011, +0.0087]"


In [20]:
rows = []
for T, page, label in ((2025, "WLDs", "Worlds 2025"), (2026, "MSI", "MSI 2026")):
    for name in FULL_VARIANTS:
        n, brier, acc = series_brier([r for r in full[(T, name)] if r["page"] == page])
        rows.append({"torneo": label, "variante": name, "series": n, "brier": brier, "aciertos": acc})
pd.DataFrame(rows).pivot(index="variante", columns="torneo", values=["brier", "aciertos"])

brier             aciertos            
torneo                MSI 2026 Worlds 2025 MSI 2026 Worlds 2025
variante                                                       
A (cruces) + roles      0.1559      0.2014     0.80       0.705
A + roles               0.2044      0.1859     0.65       0.795
Base (cruces) + roles   0.1569      0.2042     0.80       0.682
Producción (elo-2)      0.1569      0.1842     0.80       0.727

In [21]:
def cross_verdict():
    yearly = cross_results[(cross_results.partidos == "entre regiones") & (cross_results.prueba != "2022–2026")
                           & (cross_results.variante == "A (cruces)")]
    pooled = cross_results[(cross_results.partidos == "entre regiones") & (cross_results.prueba == "2022–2026")
                           & (cross_results.variante == "A (cruces)")].iloc[0]
    intl = full_cross[(full_cross.partidos == "internacionales") & (full_cross.prueba == "2025–2026")
                      & (full_cross.variante == "A (cruces) + roles")].iloc[0]
    better = int((yearly["Δ vs Base (cruces)"] < 0).sum())
    c1, c2 = pooled["IC alto"] < 0, better >= 4
    c3 = intl["Δ vs ref"] < 0 and not intl["IC bajo"] > 0
    return {"1. IC cruces < 0": c1, "2. años que mejora": f"{better}/5", "2. ≥ 4/5": c2,
            "3. internacionales vs producción": c3, "PASA": c1 and c2 and c3}

pd.DataFrame([cross_verdict()])

,1. IC cruces < 0,2. años que mejora,2. ≥ 4/5,3. internacionales vs producción,PASA
0,True,3/5,False,False,False


## 7. Conclusión

**El modelo del sitio queda como está (`elo-2`).** A mejora de verdad, pero no donde el sitio la necesita.

**A. Arrastre entre temporadas: real para las ligas, invisible en los internacionales.**
- Solo con el Elo, mejora los 5 años de prueba (2022–2026) sin excepción: Brier −0.0033, IC [−0.0042, −0.0023]. Pasa la regla.
- Con la forma por rol encima (Fase 2) se mantiene: −0.0023 contra Base + roles, IC entero por debajo de 0. No se pisa con la forma por rol.
- El `carry` elegido ronda 0.5–0.8: conviene arrastrar más de la mitad de la temporada anterior. Con A el K óptimo baja (40 → 30): un buen punto de partida reemplaza parte de lo que el Elo tenía que aprender rápido.
- **Pero contra producción empeora los internacionales**, que son justo los partidos de Worlds. El motivo es K (ver abajo).

**B. Región compartida: se descarta.** Mejora 1 de 5 años y empeora los cruces entre regiones (+0.006, IC entero por encima de 0). En 2024 pide `pool` = 1.5 y se rompe: con pocos cruces por año, el rating de región persigue el ruido de un par de torneos.

**A+B:** no le gana a A sola (IC cruza el 0). Por la regla queda la más simple.

**Variante post-hoc, A ajustada con los cruces (como producción): no pasa.** Mejora los cruces 2022–2026 juntos (−0.0028, IC entero por debajo de 0), pero solo en 3 de 5 años, y contra producción en los internacionales de 2025–2026 queda peor (+0.0028, IC cruza el 0). Además el `carry` elegido salta de 0 a 0.7 según la ventana: con ~200 cruces por año el dato está poco determinado.

**Lo más interesante: hay una tensión entre ligas e internacionales.**
- Ajustado con *todos* los games, el Elo elige K = 30–40: le conviene seguir la forma reciente, porque dentro de una liga los equipos se enfrentan seguido y la forma cuenta.
- Ajustado con los *cruces*, elige K = 4–10: entre regiones conviene un rating estable, porque son pocos partidos y una racha en la liga local dice poco de cómo te va contra otra región.
- Producción ya está del lado correcto para Worlds (K = 10, ajustado con cruces). Cualquier mejora para las ligas que sube K le cuesta en los internacionales.

**Limitaciones:**
- Los internacionales son ~200–300 games por año: una mejora real de 0.002 de Brier es más chica que el ruido de 2 años de internacionales (IC de ±0.005). Puede haber una mejora chica que estos datos no llegan a ver.
- En 2025 producción juega con ventaja (sus parámetros se eligieron mirando 2025). En 2026, limpio para todos, A (cruces) + roles queda apenas mejor en internacionales (−0.0021, IC cruza el 0): una pista, no evidencia.
- Fase 2 con solo 2 años de prueba: los CSV de Oracle's Elixir anteriores a 2024 no se pudieron bajar (cuota de Google Drive).
- La regla post-hoc se escribió después de ver la Fase 2: aunque hubiera pasado, habría sido evidencia débil.